# 하이브리드 최종 실행 (GCN · SAGE · GAT, 한 환경에서 한 번에)

보고서에 쓸 하이브리드 숫자를 **한 Colab 환경에서** 전부 다시 만듭니다. 환경이 다르면 같은 설정도 점수가 조금씩 달라지므로(예: `xgb_check` 0.8004 vs 0.8021), 비교표의 숫자는 모두 이 노트북 한 번의 실행에서 나와야 합니다.

| 단계 | 내용 |
|---|---|
| 1~4 | 환경 확인, 저장소(main) + Git LFS, 데이터, 설치 |
| 5 | 기준 임베딩·best_epoch 점검 |
| 6 | 코드 점검 (pytest) |
| 7 | **OOF 임베딩 생성** — 세 모델 모두 팀 표준(`make_oof_emb.py`, 5개 시간 블록 + Procrustes 정렬) |
| 8 | OOF 진단 (정렬 품질, 폴드 모델 성능) |
| 9 | **XGBoost 하이브리드 9종** (기준선 / 모델별 naive·OOF / 3개 합친 naive·OOF) |
| 10~12 | 결과 비교: 시드별 짝 비교, 구간(35~42 / 43~49), gain 비중 |
| 13 | 드라이브 백업 + 작은 결과 zip |

**런타임:** 런타임 > 런타임 유형 변경 > **GPU(T4 또는 L4)**. 예상 시간 30~50분 (대부분 7단계 GAT).

**규칙:** 설정 선택은 검증(30~34)으로만, 테스트(35~49)는 결과 보고에만 씁니다. 이 노트북은 선택을 하지 않고 정해진 설정으로 돌리기만 합니다.

In [ ]:
# 1) 환경 확인 (README에 같이 기록)
!python --version
!nvidia-smi -L || echo 'GPU 없음 -> 런타임 유형을 GPU로 바꾸세요'
import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# 2) 저장소(main) 받기 + Git LFS로 GAT 임베딩 실제 파일 받기
#    --depth 1: 예전 임베딩 버전까지 받지 않도록 최신 커밋만
import getpass, os
if not os.path.exists('/content/2026-2-HAI-ProjectTeam8'):
    tok = getpass.getpass('GitHub 토큰 (public이면 그냥 엔터): ')
    host = f'{tok}@' if tok else ''
    !git clone -q --depth 1 -b main https://{host}github.com/HanyangTechAI/2026-2-HAI-ProjectTeam8.git /content/2026-2-HAI-ProjectTeam8
    del tok, host
%cd /content/2026-2-HAI-ProjectTeam8
!git lfs version || (apt-get -qq install git-lfs > /dev/null && git lfs version)
!git lfs install --local > /dev/null && git lfs pull
!git log --oneline -3
!ls models hybrid

위 `ls`에 `models/gat.py`, `hybrid/make_oof_emb.py`, `hybrid/run_hybrid.py`가 모두 보여야 합니다.

In [ ]:
# 3) 데이터 복사 (드라이브 -> data/elliptic). 바로가기 위치가 달라도 찾도록 후보 경로를 순서대로 시도
import os, shutil
from google.colab import drive
drive.mount('/content/drive')
CANDIDATES = [
    "/content/drive/MyDrive/elliptic",                    # 내 드라이브 루트에 바로가기를 둔 경우
    "/content/drive/MyDrive/Colab Notebooks/elliptic",   # 마린 방식
]
NEED = ["elliptic_txs_features.csv", "elliptic_txs_classes.csv", "elliptic_txs_edgelist.csv"]
src = next((p for p in CANDIDATES if all(os.path.exists(f"{p}/{f}") for f in NEED)), None)
if src is None:
    for p in CANDIDATES:
        print(p, "->", os.listdir(p) if os.path.isdir(p) else "없음")
    raise FileNotFoundError("elliptic CSV 3개를 못 찾음: 공유받은 폴더를 '드라이브에 바로가기 추가' 했는지 확인")
os.makedirs("data/elliptic", exist_ok=True)
for f in NEED:
    shutil.copy(f"{src}/{f}", f"data/elliptic/{f}")
print("복사 완료:", src)
!ls -la data/elliptic

In [ ]:
# 4) 설치: PyG + 공식 베이스라인과 같은 xgboost 버전
!pip install -q torch-geometric "xgboost>=3.4.1,<3.5" pytest scipy
import xgboost, torch_geometric
print('xgboost', xgboost.__version__, '(공식 베이스라인: 3.4.1)', '| torch_geometric', torch_geometric.__version__)

In [ ]:
# 5) 설정 + 기준 임베딩·best_epoch 점검
import numpy as np, pandas as pd
RES = 'results_final'                 # 이 노트북의 모든 결과 (기존 results/ 파일은 건드리지 않음)
OOF = f'{RES}/emb_oof'
SEEDS = [0, 1, 2, 3, 4]
GNNS = ['gcn', 'sage', 'gat']

# 각 GNN의 최종 설정 (README 기준). epochs=None이면 results/metrics/{model}.csv의 best_epoch를 씀
GNN_CFG = {
    'gcn':  dict(lr=0.01,  dropout=0.5, wd=5e-4, epochs=None),
    'sage': dict(lr=0.005, dropout=0.3, wd=1e-3, epochs=None),
    # GAT: 저장소의 기준 임베딩은 '원래 실행'에서 나왔고 gat.csv는 재학습 실행이라 best_epoch가 다름
    #      -> 기준 임베딩과 같은 실행의 에폭을 직접 지정 (README GAT 하이브리드 절)
    'gat':  dict(lr=0.01,  dropout=0.3, wd=5e-4, epochs=[100, 330, 159, 405, 122]),
}
EXPECT_EPOCHS = {'gcn': [144, 167, 203, 306, 224], 'sage': [123, 138, 128, 232, 121]}

for m in GNNS:
    for s in SEEDS:
        p = f'results/emb/emb_{m}_seed{s}.npy'
        with open(p, 'rb') as f:
            assert not f.read(64).startswith(b'version https://git-lfs'), f'{p} 가 LFS 포인터입니다 -> 2번 셀의 git lfs pull 확인'
        e = np.load(p, mmap_mode='r')
        assert e.shape == (203769, 64) and np.isfinite(e[:1000]).all(), (p, e.shape)
    mt = pd.read_csv(f'results/metrics/{m}.csv', dtype={'threshold_type': str})
    t = mt[(mt.split == 'test') & (mt.threshold_type == 'fixed_0.5')].sort_values('seed')
    ep = t.notes.str.extract(r'best_epoch=(\d+)')[0].astype(int).tolist()
    used = GNN_CFG[m]['epochs'] or ep
    print(f'{m:4s} 기준 임베딩 OK | 단독 test PR-AUC {t.pr_auc.mean():.3f} | metrics best_epoch {ep} | OOF에 쓸 에폭 {used}')
    if m in EXPECT_EPOCHS:
        assert ep == EXPECT_EPOCHS[m], f'{m}: best_epoch가 README와 다릅니다 -> 임베딩과 지표가 같은 실행인지 확인'

In [ ]:
# 6) 코드 점검 (가짜 데이터, 몇 십 초). 전부 passed여야 다음으로 진행
!python -m pytest -q -p no:cacheprovider tests/test_hybrid.py tests/test_oof.py

## 7. OOF 임베딩 생성 (GPU)
세 모델 모두 같은 방식입니다: 시간 단계 1~34를 5개 블록(1~7 / 8~14 / 15~21 / 22~28 / 29~34)으로 나누고, 블록마다 나머지 블록의 정답으로 GNN을 학습해 그 블록의 임베딩을 만든 뒤, 라벨 없는 노드로 기준 임베딩 좌표계에 정렬(Procrustes)합니다. 테스트(35~49)는 기준 임베딩을 그대로 씁니다.

모델 × 시드 5개 × 블록 5개 = 모델당 25번 학습. GAT가 가장 오래 걸립니다.

In [ ]:
# 7) OOF 임베딩: results_final/emb_oof/emb_oof_{model}_seed{n}.npy + oof_report_{model}.csv
import subprocess, time
oof_sec = {}
for m in GNNS:
    c = GNN_CFG[m]
    cmd = ['python', '-m', 'hybrid.make_oof_emb', '--model', m, '--lr', str(c['lr']), '--dropout', str(c['dropout']),
           '--weight-decay', str(c['wd']), '--seeds', *map(str, SEEDS), '--out-dir', OOF]
    if c['epochs']:
        cmd += ['--best-epochs', *map(str, c['epochs'])]
    print('$', ' '.join(cmd)); t0 = time.time()
    subprocess.run(cmd, check=True)
    oof_sec[m] = time.time() - t0
    print(f'== {m} 완료: {oof_sec[m] / 60:.1f}분\n')
!ls -la {OOF}

In [ ]:
# 8) OOF 진단: 정렬 품질과 폴드 모델 성능 (시드 평균)
#    cos_raw: 정렬 전 / cos_aligned_unlabeled: 정렬 후(정렬에 쓴 노드) / cos_aligned_labeled: 정렬 후(정렬에 안 쓴 라벨 노드 -> 높아야 정렬이 일반화된 것)
diag = []
for m in GNNS:
    r = pd.read_csv(f'{OOF}/oof_report_{m}.csv')
    by_fold = r.groupby('steps', sort=False)[['cos_raw', 'cos_aligned_unlabeled', 'cos_aligned_labeled', 'oof_pr_auc']].mean()
    print(f'== {m} (폴드별, 시드 평균)'); print(by_fold.round(3).to_string()); print()
    diag.append(dict(model=m, cos_raw=r.cos_raw.mean(), cos_aligned_labeled=r.cos_aligned_labeled.mean(),
                     oof_pr_auc_pooled=r.groupby('seed').oof_pr_auc_pooled.first().mean(),
                     fold1_oof_pr_auc=r[r.fold == 1].oof_pr_auc.mean(), oof_minutes=oof_sec.get(m, np.nan) / 60))
print(pd.DataFrame(diag).round(3).to_string(index=False))

## 9. XGBoost 하이브리드 9종
XGBoost 설정은 베이스라인이 검증으로 고른 1등 설정(`results/xgb_tuning.csv`)으로 고정하고 **입력만** 바꿉니다. 임베딩 시드 n ↔ XGBoost 시드 n.

| 이름 | 입력 | 의미 |
|---|---|---|
| `xgb_check` | 원본 165 | 기준선 (이 환경에서 재현) |
| `hybrid_{gcn,sage,gat}_naive` | 원본 + 기존 임베딩 | 누수 있음 (비교용) |
| `hybrid_{gcn,sage,gat}` | 원본 + OOF 임베딩 | 모델별 기여 |
| `hybrid_all_naive` | 원본 + 3개 기존 임베딩 (357차원) | 누수 있음 (비교용) |
| **`hybrid_all`** | 원본 + 3개 OOF 임베딩 (357차원) | **본 실험 (계획서)** |

In [ ]:
# 9) 하이브리드 실행 (XGBoost는 CPU)
EXPS = [('xgb_check', ['--mode', 'raw'])]
for m in GNNS:
    EXPS += [(f'hybrid_{m}_naive', ['--emb-models', m]),
             (f'hybrid_{m}', ['--emb-models', m, '--emb-kind', 'oof', '--emb-dir', OOF])]
EXPS += [('hybrid_all_naive', ['--emb-models', *GNNS]),
         ('hybrid_all', ['--emb-models', *GNNS, '--emb-kind', 'oof', '--emb-dir', OOF])]
for name, a in EXPS:
    cmd = ['python', '-m', 'hybrid.run_hybrid', *a, '--seeds', *map(str, SEEDS), '--results-dir', RES]
    print('$', ' '.join(cmd)); t0 = time.time()
    out = subprocess.run(cmd, check=True, capture_output=True, text=True).stdout
    assert f'결과 이름: {name}' in out, out[:500]
    print('\n'.join(l for l in out.splitlines() if l.startswith(('[seed', 'PR-AUC', '격차'))))
    print(f'-- {name}: {time.time() - t0:.0f}초\n')

## 10~12. 결과 비교
비교 기준은 항상 **같은 환경의 `xgb_check`**, 같은 시드끼리 짝지은 차이입니다. 차이가 시드 간 표준편차보다 작거나 시드 일부에서만 양수면 "개선"이라고 쓰지 않습니다.

**테스트 구간:** 테스트(35~49)에서 빼는 구간은 없습니다. 팀 결정대로 전체(35~49) 점수를 보고하고, 12번 셀에서 **35~42 / 43~49로 나눠** 다시 계산합니다 (43단계 무렵 다크마켓 폐쇄 이후 분포 변화). 해석은 35~42 중심입니다.

In [ ]:
# 10) 테스트 PR-AUC: 시드별 + 짝 비교 (OOF 하이브리드 vs xgb_check)
from scipy import stats
pd.set_option('display.width', 220)
NAMES = [n for n, _ in EXPS]
def pick(name, split='test', tt='fixed_0.5'):
    m = pd.read_csv(f'{RES}/metrics/{name}.csv', dtype={'threshold_type': str})
    return m[(m.split == split) & (m.threshold_type == tt)].sort_values('seed').set_index('seed')

base = pick('xgb_check')
by_seed = pd.DataFrame({n: pick(n).pr_auc for n in NAMES})
print('== 테스트 PR-AUC (시드별) =='); print(by_seed.round(4).to_string()); print()

rows = []
for n in NAMES:
    t, v, tr = pick(n), pick(n, 'val', 'val_opt'), pick(n, 'train', 'val_opt')
    d = (t.pr_auc - base.pr_auc).values
    rows.append({'실험': n, 'test PR-AUC': f'{t.pr_auc.mean():.4f} ± {t.pr_auc.std():.4f}',
                 'Δ vs xgb_check': f'{d.mean():+.4f} ± {d.std(ddof=1):.4f}' if n != 'xgb_check' else '-',
                 '기준선보다 높은 시드': f'{int((d > 0).sum())}/{len(d)}' if n != 'xgb_check' else '-',
                 '대응 t검정 p': f'{stats.ttest_rel(t.pr_auc, base.pr_auc).pvalue:.3f}' if n != 'xgb_check' else '-',
                 'test ROC-AUC': round(t.roc_auc.mean(), 4), 'val PR-AUC': round(v.pr_auc.mean(), 4),
                 'train−test': round(tr.pr_auc.mean() - t.pr_auc.mean(), 4)})
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print('\n참고: 공식 xgb 베이스라인(results/metrics/xgb.csv) test PR-AUC', round(pd.read_csv('results/metrics/xgb.csv', dtype={'threshold_type': str}).query("split=='test' and threshold_type=='fixed_0.5'").pr_auc.mean(), 4))

In [ ]:
# 11) F1 / 정밀도 / 재현율 (테스트, 두 임계값)
rows = []
for n in NAMES:
    for tt in ('fixed_0.5', 'val_opt'):
        t = pick(n, 'test', tt)
        rows.append(dict(실험=n, 임계값=tt, F1=f'{t.f1.mean():.3f} ± {t.f1.std():.3f}',
                         정밀도=round(t.precision.mean(), 3), 재현율=round(t.recall.mean(), 3)))
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# 12) 테스트 구간 분리 PR-AUC (35~42 / 43~49) + gain 비중
from sklearn.metrics import average_precision_score as AP
from common.data import load_data
d = load_data('data/elliptic', standardize=False)
SEG = {'35~42': (35, 42), '43~49': (43, 49)}
masks = {k: d.test_mask & (d.time_step >= a) & (d.time_step <= b) for k, (a, b) in SEG.items()}
print('구간별 라벨/불법 수:', {k: (int(m.sum()), int((d.y[m] == 1).sum())) for k, m in masks.items()})
seg_rows, seg_val = [], {}
for n in NAMES:
    r = {'실험': n}
    for k, m in masks.items():
        v = np.array([AP(d.y[m], np.load(f'{RES}/pred/pred_{n}_seed{s}.npy')[m]) for s in SEEDS])
        seg_val[n, k] = v
        r[f'PR-AUC {k}'] = f'{v.mean():.4f} ± {v.std(ddof=1):.4f}'
    # 팀 결정: 결과 해석은 35~42 중심 -> 이 구간에서도 기준선과 시드별 짝 비교
    if n != 'xgb_check':
        dd = seg_val[n, '35~42'] - seg_val['xgb_check', '35~42']
        r['Δ 35~42 vs xgb_check'] = f'{dd.mean():+.4f} ± {dd.std(ddof=1):.4f} ({int((dd > 0).sum())}/{len(dd)}시드 높음)'
    else:
        r['Δ 35~42 vs xgb_check'] = '-'
    imp = pd.read_csv(f'{RES}/hybrid_importance_{n}.csv').groupby('group').gain_share.mean()
    r['gain 비중'] = ' / '.join(f'{g} {v:.0%}' for g, v in imp.items())
    seg_rows.append(r)
seg = pd.DataFrame(seg_rows)
print(seg.to_string(index=False))

final = summary.merge(seg, on='실험')
final.to_csv(f'{RES}/final_summary.csv', index=False)
by_seed.to_csv(f'{RES}/final_test_prauc_by_seed.csv')
print('\n저장:', f'{RES}/final_summary.csv', f'{RES}/final_test_prauc_by_seed.csv')

In [ ]:
# 13) 환경 기록 + 드라이브 백업 + 작은 결과 zip (OOF 임베딩 npy는 용량이 커서 드라이브에만)
import subprocess, shutil, datetime
from google.colab import files
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()
commit = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
with open(f'{RES}/hybrid_env.txt', 'w') as f:
    f.write(f'하이브리드 최종 실행 환경 (이 폴더의 모든 결과는 이 한 번의 실행에서 생성)\n'
            f'- 생성일: {datetime.date.today()} / 저장소 main {commit}\n- GPU: {gpu}\n'
            f'- torch {torch.__version__} / torch_geometric {torch_geometric.__version__} / xgboost {xgboost.__version__}\n'
            f'- GNN 설정(OOF): {GNN_CFG}\n- OOF 소요(분): { {k: round(v / 60, 1) for k, v in oof_sec.items()} }\n')
print(open(f'{RES}/hybrid_env.txt').read())

dst = '/content/drive/MyDrive/hybrid_final'
shutil.copytree(RES, dst, dirs_exist_ok=True)
print('드라이브 백업:', dst)
shutil.copytree(RES, 'out_small', ignore=shutil.ignore_patterns('emb_oof_*.npy'), dirs_exist_ok=True)
shutil.make_archive('hybrid_final_small', 'zip', 'out_small')
files.download('hybrid_final_small.zip')